# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tayyabp0g/Fly_Rank_MachineLearning_Ass_2/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane 2 — Refresh / Content Opportunity Scoring.** The task type is **ranking**: the product is an *ordered list* that answers "which pages should a person review first?". To build that list I will *score* every page (a risk/opportunity score) and sort by it. A binary classifier is a possible *tool* for producing the score (its probability becomes the score), but the thing I deliver and evaluate is the ranked queue, not a yes/no verdict per page. It is **not clustering** — I am not looking for page types; I need an order.

**How the lane maps onto the ML loop**

| Step of the loop | In my lane |
|---|---|
| Data | Starter CSV now (30,000 pages, 32 pseudonymized clients); the warehouse release later |
| Unit of analysis | One page (one pseudonymized content item) |
| Features | Observable signals known *before* the decision: exposure, position, CTR, age, freshness, content properties |
| Target / proxy | Starter proxy: "declining" (section 2); stronger future-window target planned |
| Method | Transparent rule baseline first, then a model that must beat it |
| Evaluation | Precision@K against the base rate, validated by client (section 3) |
| **Action** | An editor reviews the top K pages and chooses: refresh the content, rewrite title/meta when CTR is low for that position, add depth, or monitor / leave alone |
| Feedback | What the editor decides — and what later happens to the page — can become better labels. Careful: pages we review get changed, which changes future data (a feedback loop) |

The code cell below only sketches the **shape** of one queue (rank, score, reason code, suggested action). The score is a placeholder (visibility), not a model.

In [1]:
import os, sys, subprocess
import numpy as np
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)

# Find the repo root (Colab: clone the public starter repo, which ships the same starter CSV).
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.isdir("data/raw"):
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
                        "flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

raw = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Starter data: {len(raw):,} pages x {raw.shape[1]} columns, {raw['client_id'].nunique()} pseudonymized clients")

# Sketch of the OUTPUT SHAPE only (not a model): what one row of the ranked review queue will look like.
# The score here is just visibility (impressions) so the sketch uses no label and no leakage.
sketch = (raw.sort_values("impressions_90d", ascending=False).head(5)
             [["content_id", "impressions_90d", "avg_position", "ctr"]]
             .reset_index(drop=True))
sketch.insert(0, "rank", sketch.index + 1)
sketch["score"] = sketch["impressions_90d"]                       # placeholder score
sketch["reason_code"] = "high_visibility (placeholder)"
sketch["suggested_action"] = "review for refresh / CTR (placeholder)"
sketch

Starter data: 30,000 pages x 44 columns, 32 pseudonymized clients


,rank,content_id,impressions_90d,avg_position,ctr,score,reason_code,suggested_action
0,1,content_5fe46e04994d,517715,4.2,0.14,517715,high_visibility (placeholder),review for refresh / CTR (placeholder)
1,2,content_aaef01a50def,517109,5.4,0.25,517109,high_visibility (placeholder),review for refresh / CTR (placeholder)
2,3,content_8c19996aa890,509252,2.5,0.15,509252,high_visibility (placeholder),review for refresh / CTR (placeholder)
3,4,content_2cb567c3c89b,497727,22.2,0.10,497727,high_visibility (placeholder),review for refresh / CTR (placeholder)
4,5,content_4c36c775b818,463103,2.3,0.41,463103,high_visibility (placeholder),review for refresh / CTR (placeholder)


## 2. Target or proxy

**What I would predict:** whether a page is *in decline*, so that pages with a higher chance can be put higher in the queue.

**Where the label comes from — a defined rule, not an observed future outcome.** The starter label `y_proxy_declining` is 1 when `trend_direction == "down"`. The code below rebuilds it from its raw inputs and confirms it is exactly "impressions in the last 30 days fell by more than 20% compared with the 30 days before". So it is a **proxy**: a bucket computed by a rule from observed numbers, in the same window as my features.

**Three things I found that shape how I will use it**
1. **The "not declining" group is a mix.** It holds `stable` (5,962), `up` (4,388), `new` (2,236) and `flat` (1,152) pages. `new` pages (no impressions in the earlier window) and `flat` pages (none in either window) cannot decline by definition, yet the starter treats them as 0. Whether to keep or drop them is a choice I will make and defend in the data contract (Week 3).
2. **Feature window and label window overlap.** The last-30-day and previous-30-day numbers sit *inside* the 90-day totals (they make up about 57% of a page's 90-day impressions at the median). So 90-day features such as `impressions_90d` or `days_with_impressions` partly "see" the outcome. Section 5 measures how much this matters.
3. **Columns that are never features:** `trend_direction`, `trend_pct`, and the `*_last_30d` / `*_prev_30d` columns (they are the label's direct inputs), plus the ID columns.

**Stronger target I plan to use on the warehouse (a spec only, not built here):** features from a prior 90 days, outcome measured in the *next* 30 days, with a size threshold, a minimum-volume rule and a persistence check (printed below). That is a real *future* outcome, so features and target never overlap. If it cannot be built cleanly I will say so and keep the proxy, clearly labeled as a proxy.

In [2]:
# --- The proxy target, rebuilt from its raw inputs so we can see exactly where it comes from ---
d = raw.copy()
prev, last = d["impressions_prev_30d"], d["impressions_last_30d"]
d["trend_pct_rebuilt"] = np.where(prev > 0, (last - prev) / prev * 100, np.nan)
d["y_proxy_declining"] = d["trend_direction"].eq("down").astype(int)          # the starter label

# It is defined by a RULE on two observed numbers: "down" means last-30d impressions fell by MORE than 20%
rule_down = (prev > 0) & (d["trend_pct_rebuilt"] < -20)
print("Label equals the -20% rule on (last 30d vs previous 30d impressions):",
      bool((rule_down.astype(int) == d["y_proxy_declining"]).all()))
print("\nTarget column distribution (y_proxy_declining):")
print(d["y_proxy_declining"].value_counts().rename({1: "declining (1)", 0: "not declining (0)"}).to_string())
print("\nWhat sits inside the 'not declining (0)' group:")
print(d.loc[d["y_proxy_declining"] == 0, "trend_direction"].value_counts().to_string())

# --- Feature window vs label window: they overlap ---
print("\nAre last-30d + previous-30d impressions always inside the 90d total?",
      bool((d["impressions_90d"] >= last + prev).all()))
print(f"Median share of a page's 90d impressions that fall in those two 30-day windows: "
      f"{((last + prev) / d['impressions_90d']).median():.0%}")

# --- Sketch of the STRONGER target planned for the warehouse (a spec, not built here) ---
future_target_spec = {
    "feature_window": "daily rows in the prior 90 days (ends at the decision date)",
    "target_window":  "the NEXT 30 days after the decision date (never overlaps the feature window)",
    "y_future_decline": "1 if next-30d impressions < 80% of the last 30 days of the feature window",
    "minimum_volume": "at least 100 impressions in the feature window (rules out noise)",
    "persistence":    "also below the threshold in the 2nd half of the target window (rules out blips)",
    "checks":         "per-client history start, seasonality vs site-level trend, sibling-page consolidation",
}
for k, v in future_target_spec.items():
    print(f"{k:>17}: {v}")

Label equals the -20% rule on (last 30d vs previous 30d impressions): True

Target column distribution (y_proxy_declining):
y_proxy_declining
declining (1)        16262
not declining (0)    13738

What sits inside the 'not declining (0)' group:
trend_direction
stable    5962
up        4388
new       2236
flat      1152

Are last-30d + previous-30d impressions always inside the 90d total? True
Median share of a page's 90d impressions that fall in those two 30-day windows: 57%
   feature_window: daily rows in the prior 90 days (ends at the decision date)
    target_window: the NEXT 30 days after the decision date (never overlaps the feature window)
 y_future_decline: 1 if next-30d impressions < 80% of the last 30 days of the feature window
   minimum_volume: at least 100 impressions in the feature window (rules out noise)
      persistence: also below the threshold in the 2nd half of the target window (rules out blips)
           checks: per-client history start, seasonality vs site-leve

## 3. Success metric

**Primary metric: Precision@K, with K = 20 and K = 50, reported as lift over the base rate and validated with client-holdout.** Precision@K asks: of the K pages the queue puts first, how many carry the label? It matches how the list is used — a person only reviews K pages — so generic accuracy would be the wrong measure.

**What number means "good"?** Not an absolute number. "Good" means the queue beats **both** (a) the base rate — what a random queue gets (0.542 on the whole file) — **and** (b) the best simple rule, by more than the noise, and does so across client folds and not on a single lucky split.

**Why I will not over-read small gaps.** The code shows the standard error of Precision@K is about 0.07 at K = 50 and 0.11 at K = 20, so differences below roughly 0.14 and 0.22 are inside the noise. I will report this next to every comparison.

**What the check below shows.** Two ultra-simple rankings set the scale. "Most visible first" gets Precision@50 of 0.42, which is *below* a random queue (lift 0.77): visibility alone is a bad way to order a decline-review list. "Youngest content first" gets 0.68 (lift 1.25), which is only about at the edge of the noise band. Whatever I build has to beat the better of these and the base rate.

**Secondary metrics:** an impressions-weighted version (how much of the top-50's exposure sits on declining pages — a missed high-exposure page costs more), and AUC as a steadier overall check. *Note:* the numbers in this notebook use the whole file to set scale; my real evaluation will use the lane slice and client-holdout.

In [3]:
def precision_at_k(scores, labels, k):
    """Of the top-k pages by score, what fraction carry the label? (ties broken by original order)"""
    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())

y = d["y_proxy_declining"].values
base = y.mean()
print(f"Base rate (what a random queue gets on average): {base:.3f}")
for k in (20, 50):
    se = np.sqrt(base * (1 - base) / k)
    print(f"  K={k}: standard error of Precision@K is about {se:.3f}  "
          f"-> differences smaller than ~{2*se:.2f} are inside the noise")

# Two ultra-simple 'rules' to show what numbers mean (these are NOT my baseline, just scale-setters)
rules = {
    "rank by impressions (most visible first)": d["impressions_90d"].values,
    "rank by youngest content first":           -d["content_age_days"].values,
}
rows = []
for name, score in rules.items():
    row = {"ranking": name}
    for k in (20, 50):
        p = precision_at_k(score, y, k)
        row[f"P@{k}"] = round(p, 3); row[f"lift@{k}"] = round(p / base, 2)
    # impressions-weighted view: what share of the top-50's impressions sit on declining pages?
    order = np.argsort(-score, kind="stable")[:50]
    row["impr-weighted@50"] = round(d["impressions_90d"].values[order][y[order] == 1].sum()
                                    / d["impressions_90d"].values[order].sum(), 3)
    rows.append(row)
print()
print(pd.DataFrame(rows).to_string(index=False))

Base rate (what a random queue gets on average): 0.542
  K=20: standard error of Precision@K is about 0.111  -> differences smaller than ~0.22 are inside the noise
  K=50: standard error of Precision@K is about 0.070  -> differences smaller than ~0.14 are inside the noise

                                 ranking  P@20  lift@20  P@50  lift@50  impr-weighted@50
rank by impressions (most visible first)  0.45     0.83  0.42     0.77             0.443
          rank by youngest content first  0.65     1.20  0.68     1.25             0.423


## 4. The unit of analysis, as a real dataframe

**One row = one page** (one pseudonymized content item). The lane slice keeps pages with at least 500 impressions in 90 days (a threshold I chose, not a universal truth), which gives **16,726 rows**. The last column is the proxy target. The table below shows 8 of those rows.

Grain check results: `content_id` is unique, so there is exactly one row per page. Pages per client range from 4 to 5,222 in this slice, so a few clients dominate. That is why train and test must be split **by client** — otherwise a model could memorize a big client's habits and look better than it is. In this slice 59.6% of pages carry the proxy label (0.542 on the full file).

IDs here are pseudonyms; they are used only to identify rows and group them, never as features.

In [4]:
# Lane 2 slice: pages with enough exposure to matter (>= 500 impressions in 90 days; a threshold I chose)
lane = d[d["impressions_90d"] >= 500].copy()

cols = ["content_id", "client_id", "content_type", "position_tier", "avg_position", "ctr",
        "impressions_90d", "content_age_days", "days_since_last_update", "y_proxy_declining"]
print(f"Lane slice: {len(lane):,} rows (one row = ONE PAGE = one pseudonymized content item)")
display(lane[cols].head(8).reset_index(drop=True))

# Grain check: one row per page?
print("\ncontent_id unique (one row per page):", lane["content_id"].is_unique)
per_client = lane.groupby("client_id").size()
print(f"Pages per client in the slice: min {per_client.min()}, median {int(per_client.median())}, max {per_client.max()}"
      f"  (so clients differ a lot in size -> split train/test BY CLIENT)")
print(f"Target base rate in the slice: {lane['y_proxy_declining'].mean():.3f}")

Lane slice: 16,726 rows (one row = ONE PAGE = one pseudonymized content item)


,content_id,client_id,content_type,position_tier,avg_position,ctr,impressions_90d,content_age_days,days_since_last_update,y_proxy_declining
0,content_304f48230142,client_f369cb89fc,keyword article,striking,10.6,0.76,3803,187,20,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,page_3_5,20.3,0.05,15320,445,25,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,page_3_5,36.5,0.09,12581,141,20,1
3,content_331d6c4de07b,client_19581e27de,keyword article,page_1,6.2,0.49,11751,463,22,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,page_3_5,44.0,0.13,19140,263,14,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,page_1,8.5,0.03,3970,147,20,1
6,content_a63219c6e95a,client_19581e27de,keyword article,page_3_5,21.2,0.06,1724,445,22,0
7,content_5e6c160719bc,client_6208ef0f77,keyword article,page_3_5,46.0,0.09,32574,90,20,1



content_id unique (one row per page): True
Pages per client in the slice: min 4, median 137, max 5222  (so clients differ a lot in size -> split train/test BY CLIENT)
Target base rate in the slice: 0.596


## 5. Why ML beats a fixed rule here

I should be careful: this section is evidence for a *hypothesis*, not proof that ML wins.

**Why a one-line rule struggles.**
- **No single column carries the signal.** The best single column as a rule is `content_age_days`, with AUC 0.591 (0.5 = coin flip). Exposure, days with impressions, search volume and word count are all only in the 0.53–0.59 range. The information is spread across many weak signals.
- **Effects depend on context.** The share labeled declining by position tier flips with a volume floor: for `top_3` pages it is 24.1% on the whole file but 74.0% in the visible slice. (Top-3 pages in the full file mostly have very few impressions, as the data dictionary warns.) A fixed threshold on position would point the wrong way depending on volume.
- **Missingness follows content type** (some content types have no keyword data), so simple rules over those columns behave differently by type.

**What combining signals buys.** A random forest using many allowed columns, validated by client (5 folds), gets a mean AUC of **0.675** (folds from 0.633 to 0.742), against **0.591** for the best single-column rule. The spread across folds is itself a warning: a single split can flatter (the starter pipeline's one holdout reports 0.750).

**The honest caveat.** Part of that edge may be mechanical, because of the window overlap in section 2. Removing the `days_with_*` columns gives 0.660, and removing all 90-day columns that overlap the label window gives **0.604** — only about 0.01 above the best single rule. So with the *proxy* label I cannot yet claim that ML clearly beats a rule. The decisive test is the future-window label on the warehouse.

**What this means for my plan.** I will build the transparent rule baseline first, then try a model, and compare both against the base rate with client-holdout. If the model does not clearly beat the baseline under the stronger label, that will be the result I report. The queue stays decision support: a person reviews the pages and decides.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

# (a) The pattern is an interaction, not a single threshold: decline rate by position tier x content type
pivot = lane.pivot_table(index="position_tier", columns="content_type", values="y_proxy_declining",
                         aggfunc="mean")
counts = lane.pivot_table(index="position_tier", columns="content_type", values="y_proxy_declining",
                          aggfunc="size")
print("Share labeled declining (cells with fewer than 100 pages hidden):")
print((pivot.where(counts >= 100) * 100).round(1).to_string())

# (a2) The same tier effect flips with a volume floor: all pages vs the visible lane slice
flip = pd.DataFrame({
    "all pages": d.groupby("position_tier")["y_proxy_declining"].mean() * 100,
    "visible slice (>=500 impressions)": lane.groupby("position_tier")["y_proxy_declining"].mean() * 100,
}).round(1)
print("\nShare labeled declining by position_tier (%), with and without a volume floor:")
print(flip.sort_values("all pages").to_string())

# (b) How good is the BEST single column used as a one-line rule? (AUC: 0.5 = coin flip, 1.0 = perfect)
single_cols = ["search_volume", "competition", "cpc", "word_count", "char_count", "impressions_90d",
               "clicks_90d", "sessions_90d", "days_with_impressions", "days_with_sessions",
               "content_age_days", "days_since_last_update", "ctr", "avg_position",
               "engagement_rate", "scroll_rate"]
yy = d["y_proxy_declining"].values
single = {}
for c in single_cols:
    a = roc_auc_score(yy, d[c].fillna(d[c].median()))
    single[c] = max(a, 1 - a)                                  # direction does not matter for a rule
best_col, best_auc = max(single.items(), key=lambda t: t[1])
print(f"\nBest single-column rule: {best_col}  (AUC {best_auc:.3f})")

# (c) A model that combines many weak signals, validated BY CLIENT (GroupKFold, 5 folds)
def build_X(drop=()):
    X = d[[c for c in single_cols if c not in drop]].copy()
    for c in ["word_count", "char_count", "search_volume", "cpc", "competition"]:
        X["has_" + c] = d[c].notna().astype(int)               # missingness follows content_type -> flag it
    X = X.fillna(-1)
    cats = pd.get_dummies(d[["content_type", "main_intent", "position_tier", "competition_level"]].fillna("unknown"))
    return pd.concat([X, cats], axis=1)                          # NO ids, NO trend_*, NO last/prev-30d columns

def grouped_auc(X):
    aucs = []
    for tr, te in GroupKFold(n_splits=5).split(X, yy, d["client_id"]):
        m = RandomForestClassifier(n_estimators=100, min_samples_leaf=20, n_jobs=-1, random_state=42)
        m.fit(X.iloc[tr], yy[tr])
        aucs.append(roc_auc_score(yy[te], m.predict_proba(X.iloc[te])[:, 1]))
    return np.array(aucs)

variants = {
    "all allowed features":                              (),
    "minus days_with_* (touch the label window)":        ("days_with_impressions", "days_with_sessions"),
    "minus days_with_* and 90d totals (all overlap it)": ("days_with_impressions", "days_with_sessions",
                                                          "impressions_90d", "clicks_90d", "sessions_90d"),
}
print("\nRandom forest, client-holdout (5 folds), AUC:")
auc_table = []
for name, drop in variants.items():
    a = grouped_auc(build_X(drop))
    auc_table.append({"features used": name, "mean AUC": round(a.mean(), 3),
                      "worst fold": round(a.min(), 3), "best fold": round(a.max(), 3)})
print(pd.DataFrame(auc_table).to_string(index=False))

Share labeled declining (cells with fewer than 100 pages hidden):
content_type   comparison article  feedly article  keyword article
position_tier                                                     
deep                          NaN             NaN             29.2
page_1                        NaN             NaN             57.9
page_3_5                      NaN             NaN             61.3
striking                      NaN             NaN             61.6
top_3                         NaN             NaN             74.1

Share labeled declining by position_tier (%), with and without a volume floor:
               all pages  visible slice (>=500 impressions)
position_tier                                              
top_3               24.1                               74.0
deep                34.4                               29.8
page_3_5            56.2                               61.4
page_1              57.0                               57.9
striking            61.0 

                                    features used  mean AUC  worst fold  best fold
                             all allowed features     0.675       0.633      0.742
       minus days_with_* (touch the label window)     0.660       0.624      0.728
minus days_with_* and 90d totals (all overlap it)     0.604       0.562      0.684


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.